# موديل الطبيعي بتاعك — baseline + anomaly detection

أبسط طريقة شغالة: متوسط وانحراف معياري لكل مؤشر من أول أسبوع (الطبيعي بتاعه)، وبعدين أي يوم في الأسبوع التاني بعيد عن ده بشكل مستمر (يومين ورا بعض، مش قراءة واحدة شاذة) يبقى انحراف حقيقي.

الكود الفعلي في [`ai/baseline.py`](../baseline.py) — النوت بوك ده بيناديه ويرسم النتيجة. التشغيل: `pip install matplotlib` ثم شغّل كل الخلايا.

In [ ]:
import json, sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
from baseline import analyze

DATA = Path.cwd().parent.parent / "data" / "mock" / "day143.json"
readings = json.loads(DATA.read_text(encoding="utf-8"))
result = analyze(readings, crew_id="crew-1")
print(json.dumps(result, ensure_ascii=False, indent=2))

## شكل الرد
ده بالظبط شكل الرد اللي `contracts/ai.md` بيطلبه من `POST /analyze`: `baseline` (متوسط وانحراف لكل مؤشر) و`anomalies` (المؤشر، من إمتى، الاتجاه، الدرجة). `colorStillGreen` بيتحط من اللي بينادي الموديل (السيرفر)، لأنه هو اللي عارف حدود الألوان؛ هنا بس بنوضحه بالمقارنة مع `research/thresholds.json`.

In [ ]:
%run ../plot_baseline.py --out baseline.png
from IPython.display import Image
Image("baseline.png")

## اللي الرسم بيوضحه

- **النوم:** الموديل بيمسك الانحراف يومين قبل ما اللون الثابت يتغيّر من أخضر. (الخط البرتقالي قبل الأحمر)
- **النبض وهو مرتاح:** الموديل بيمسك الانحراف من أول أسبوع التاني، واللون الثابت **ما بيتغيرش خالص** طول الـ14 يوم — ده بالظبط `colorStillGreen: true` اللي الديمو بيحتاجه.
- مفيش انحراف غلط في أول أسبوع (منطقة الـbaseline الزرقا)، لأن الموديل أصلًا ما بيفحصش فيها.

خطوة بعد كده: `i2` بتلف الدالة `analyze()` في خدمة FastAPI، وبتحفظ نتيجة السيناريو كامل في `data/precomputed/analyze.json` كبديل لو الخدمة وقعت.